# Install Required Libraries





In [ ]:
!pip install duckdb pandas pyarrow


* We have installed required libraries to create a database and store all the
dataset files for faster access.

# Step 1A: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

We have mounted Google Drive to access the MIMIC-IV heart failure dataset files stored in the project folder.

# Step 1B: Set Data Path and Verify Files

In [ ]:
DATA_PATH = "/content/drive/MyDrive/DataScience_Capstone_Project/"

import os
print(os.listdir(DATA_PATH))


* Defined the project data path and lists all available CSV files and database to confirm the dataset is accessible.

# Step 1C: Loaded All CSV Files into DuckDB Database

In [ ]:
import duckdb
import os

DATA_PATH = "/content/drive/MyDrive/DataScience_Capstone_Project/"
DB_PATH = DATA_PATH + "hf_project.duckdb"

con = duckdb.connect(DB_PATH)

csv_files = [f for f in os.listdir(DATA_PATH) if f.endswith(".csv")]

for file in csv_files:
    table_name = file.replace(".csv", "")
    print(f"Loading {file} as table: {table_name}")

    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{DATA_PATH}{file}', IGNORE_ERRORS=true);
    """)

print("Database created at:", DB_PATH)


* Connected to a DuckDB database on Google Drive and loads all 11
CSV files as tables using `read_csv_auto`. This creates a local analytical database for efficient SQL-based querying throughout the project.

# Step 1D: Verify Loaded Tables

In [ ]:
con.execute("SHOW TABLES").fetchdf()


Listed all tables currently stored in the DuckDB database to confirm successful data loading. The database contains 15 tables including raw data tables and derived feature tables from previous pipeline steps.

# Step 1E: Display Table Names

In [ ]:

tables = con.execute("SHOW TABLES").fetchdf()
print(tables)

Printed the full list of table names in a readable format to verify the database schema before proceeding to cohort construction.

# Step 2A: Create Heart Failure Cohort with 30-Day Readmission Label

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE hf_labeled AS

    WITH hf_admissions AS (
        -- Get all HF admissions with valid admit/discharge times
        SELECT
            a.subject_id,
            a.hadm_id,
            a.admittime,
            a.dischtime
        FROM heart_diagnoses hd
        JOIN admissions a ON hd.hadm_id = a.hadm_id
        WHERE a.dischtime >= a.admittime  -- remove 175 invalid rows
    )

    SELECT
        h.subject_id,
        h.hadm_id,
        h.admittime,
        h.dischtime,
        CASE
            WHEN MIN(next_a.admittime) IS NOT NULL THEN 1
            ELSE 0
        END AS readmitted_30d

    FROM hf_admissions h
    LEFT JOIN admissions next_a
        ON  h.subject_id    = next_a.subject_id
        AND next_a.admittime > h.dischtime
        AND next_a.admittime <= h.dischtime + INTERVAL 30 DAYS
        AND next_a.hadm_id  != h.hadm_id

    GROUP BY h.subject_id, h.hadm_id, h.admittime, h.dischtime
""")

# Verify
result = con.execute("""
    SELECT
        COUNT(*)                          AS total_admissions,
        COUNT(DISTINCT subject_id)        AS unique_patients,
        SUM(readmitted_30d)               AS readmitted,
        ROUND(AVG(readmitted_30d) * 100, 2) AS readmission_rate_pct
    FROM hf_labeled
""").fetchdf()

print("hf_labeled created!")
print(result.to_string(index=False))

Built the `hf_labeled` table by joining `heart_diagnoses` with `admissions`, filtering out 175 invalid rows where discharge time precedes admit time. For each HF admission, computes the binary label `readmitted_30d` (1 = patient was readmitted within 30 days of discharge). Result: 4,761 admissions from 4,289 unique patients with a 21.72% readmission rate.

# Step 2B: Checked deaths in HF cohort

In [ ]:
died_check = con.execute("""
    SELECT
        COUNT(*) AS total_hf_admissions,
        SUM(CASE
            WHEN p.dod IS NOT NULL
             AND CAST(p.dod AS DATE) <= CAST(h.dischtime AS DATE)
            THEN 1 ELSE 0
        END) AS died_during_admission,
        SUM(CASE
            WHEN p.dod IS NOT NULL
             AND CAST(p.dod AS DATE) <= CAST(h.dischtime AS DATE) + 30
            THEN 1 ELSE 0
        END) AS died_within_30d
    FROM hf_labeled h
    JOIN patients p ON h.subject_id = p.subject_id
""").fetchdf()

print(died_check.to_string(index=False))

Identified patients who died during admission (105) or within 30 days of discharge (253). These cases need to be excluded because patients who died cannot be "readmitted", including them would bias the readmission prediction model.

#Step 2C: Recreate Cohort Excluding Deaths Within 30 Days

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE hf_labeled AS

    WITH hf_admissions AS (
        SELECT a.subject_id, a.hadm_id, a.admittime, a.dischtime
        FROM heart_diagnoses hd
        JOIN admissions a ON hd.hadm_id = a.hadm_id
        JOIN patients p ON a.subject_id = p.subject_id
        WHERE a.dischtime >= a.admittime
          AND (p.dod IS NULL
               OR CAST(p.dod AS DATE) > CAST(a.dischtime AS DATE) + 30)
    )

    SELECT
        h.subject_id,
        h.hadm_id,
        h.admittime,
        h.dischtime,
        CASE
            WHEN MIN(next_a.admittime) IS NOT NULL THEN 1
            ELSE 0
        END AS readmitted_30d

    FROM hf_admissions h
    LEFT JOIN admissions next_a
        ON  h.subject_id    = next_a.subject_id
        AND next_a.admittime > h.dischtime
        AND next_a.admittime <= h.dischtime + INTERVAL 30 DAYS
        AND next_a.hadm_id  != h.hadm_id

    GROUP BY h.subject_id, h.hadm_id, h.admittime, h.dischtime
""")

# Verify
result = con.execute("""
    SELECT
        COUNT(*)                             AS total_admissions,
        COUNT(DISTINCT subject_id)           AS unique_patients,
        SUM(readmitted_30d)                  AS readmitted,
        ROUND(AVG(readmitted_30d) * 100, 2)  AS readmission_rate_pct
    FROM hf_labeled
""").fetchdf()

print(" hf_labeled recreated (deaths excluded)!")
print(result.to_string(index=False))

Recreated `hf_labeled` by excluding patients who died during admission or within 30 days of discharge. This is a critical clinical step dead patients cannot be readmitted, and leaving them in the dataset would introduce data leakage. Final cohort: 4,508 admissions, 4,074 unique patients, 971 readmitted (21.54% readmission rate).

#Step 3A: Explore All Feature Table Schemas and Samples

In [ ]:
tables = [
    'heart_labevents_first_lab',
    'heart_labevents_examination_group',
    'heart_diagnoses_all_true',
    'heart_diagnoses_all',
    'heart_diagnoses',
    'heart_procedures',
    'heart_microbiologyevents_first_micro',
    'heart_microbiologyevents',
    'patients',
    'admissions'
]

for table in tables:
    print(f"\n{'='*60}")
    print(f"=== {table} ===")
    print(f"{'='*60}")
    display(con.execute(f"DESCRIBE {table}").fetchdf())
    print(f"Rows: {con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]}")
    print(f"Sample:")
    display(con.execute(f"SELECT * FROM {table} LIMIT 3").fetchdf())

Iterates through all 10 source tables (lab events, diagnoses, procedures, microbiology events, patients, admissions) to inspect column names, data types, row counts, and sample data. This schema exploration informs which features can be engineered for the prediction model.

# Step 3B: Identify Top 30 Lab Tests by Patient Coverage

In [ ]:
lab_labels = con.execute("""
    SELECT
        label,
        examination_group,
        COUNT(*) AS total_records,
        COUNT(DISTINCT hadm_id) AS admissions_with_this_lab,
        ROUND(COUNT(DISTINCT hadm_id) * 100.0 /
              (SELECT COUNT(DISTINCT hadm_id) FROM heart_labevents_first_lab), 2)
            AS pct_coverage
    FROM heart_labevents_first_lab
    WHERE valuenum IS NOT NULL
    GROUP BY label, examination_group
    ORDER BY admissions_with_this_lab DESC
    LIMIT 30
""").fetchdf()

display(lab_labels)

Queries `heart_labevents_first_lab` to find the 30 most frequently available lab tests across HF admissions, along with their examination groups and coverage percentages. Labs like Creatinine, Urea Nitrogen, and Potassium cover ~99% of admissions, while cardiac markers like Troponin T cover only ~53%. This guides which labs to include as model features.

# Step 3C: Verify Lab Coverage in Examination Group Table

In [ ]:
exam_labs = con.execute("""
    SELECT
        label,
        COUNT(*) AS total_records,
        COUNT(DISTINCT hadm_id) AS admissions_count,
        ROUND(AVG(CASE WHEN valuenum IS NOT NULL THEN 1 ELSE 0 END) * 100, 2) AS pct_numeric
    FROM heart_labevents_examination_group
    WHERE label IN (
        'Creatinine', 'Urea Nitrogen', 'Sodium', 'Potassium', 'Glucose',
        'Hemoglobin', 'White Blood Cells', 'Platelet Count', 'Bicarbonate',
        'Calcium, Total', 'INR(PT)', 'PTT', 'Troponin T',
        'Creatine Kinase, MB Isoenzyme'
    )
    GROUP BY label
    ORDER BY admissions_count DESC
""").fetchdf()

display(exam_labs)

Cross-references the 14 selected lab tests against `heart_labevents_examination_group` (which contains ALL lab measurements per admission, not just the first). This confirms higher coverage rates and validates the decision to use this table for building time-series lab features (first, last, min, max values per admission).

# Step 3D: Engineer Lab Features from Examination Group Data

In [ ]:
LABS = [
    'Creatinine', 'Urea Nitrogen', 'Sodium', 'Potassium', 'Glucose',
    'Hemoglobin', 'White Blood Cells', 'Platelet Count', 'Bicarbonate',
    'Calcium, Total', 'INR(PT)', 'PTT', 'Troponin T',
    'Creatine Kinase, MB Isoenzyme'
]

def lab_col(lab):
    return lab.lower().replace(' ', '_').replace(',', '').replace('(', '').replace(')', '')

first_cases = ",\n        ".join([
    f"MAX(CASE WHEN fl.label = '{lab}' THEN fl.first_val END) AS {lab_col(lab)}_first"
    for lab in LABS
])
last_cases = ",\n        ".join([
    f"MAX(CASE WHEN fl.label = '{lab}' THEN fl.last_val END) AS {lab_col(lab)}_last"
    for lab in LABS
])
min_cases = ",\n        ".join([
    f"MIN(CASE WHEN fl.label = '{lab}' THEN fl.min_val END) AS {lab_col(lab)}_min"
    for lab in LABS
])
max_cases = ",\n        ".join([
    f"MAX(CASE WHEN fl.label = '{lab}' THEN fl.max_val END) AS {lab_col(lab)}_max"
    for lab in LABS
])

con.execute(f"""
    CREATE OR REPLACE TABLE lab_features AS

    WITH ranked AS (
        SELECT
            hadm_id,
            label,
            valuenum,
            charttime,
            ROW_NUMBER() OVER (PARTITION BY hadm_id, label ORDER BY charttime ASC) AS rn_first,
            ROW_NUMBER() OVER (PARTITION BY hadm_id, label ORDER BY charttime DESC) AS rn_last
        FROM heart_labevents_examination_group
        WHERE valuenum IS NOT NULL
          AND label IN ('{"','".join(LABS)}')
    ),
    lab_agg AS (
        SELECT
            hadm_id,
            label,
            MAX(CASE WHEN rn_first = 1 THEN valuenum END) AS first_val,
            MAX(CASE WHEN rn_last = 1 THEN valuenum END) AS last_val,
            MIN(valuenum) AS min_val,
            MAX(valuenum) AS max_val
        FROM ranked
        GROUP BY hadm_id, label
    )

    SELECT
        fl.hadm_id,
        {first_cases},
        {last_cases},
        {min_cases},
        {max_cases}
    FROM lab_agg fl
    GROUP BY fl.hadm_id
""")

# Verify
lab_df = con.execute("SELECT * FROM lab_features LIMIT 5").fetchdf()
print(f"Lab features: {con.execute('SELECT COUNT(*) FROM lab_features').fetchone()[0]} rows, {len(lab_df.columns)} columns")
print(f"\nColumns: {list(lab_df.columns)}")
display(lab_df)

Constructed the `lab_features` table from `heart_labevents_examination_group` using 14 clinically relevant lab tests. For each admission, extracts four temporal aggregations per lab: **first** value (admission baseline), **last** value (discharge status), **min**, and **max** (extremes during stay). This captures patient trends over the hospital stay — a key differentiator from using only first-lab values. Result: 4,748 rows × 57 columns.

# Step 3E: Engineer Clinical, Demographic, and Administrative Features

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE other_features AS

    -- Count comorbidities per admission
    WITH comorbidity_count AS (
        SELECT hadm_id, COUNT(DISTINCT icd_code) AS num_comorbidities
        FROM heart_diagnoses_all_true
        GROUP BY hadm_id
    ),

    -- Count procedures per admission
    procedure_count AS (
        SELECT hadm_id, COUNT(DISTINCT icd_code) AS num_procedures
        FROM heart_procedures
        GROUP BY hadm_id
    ),

    -- Infection flag: did patient have any microbiology event?
    infection_flag AS (
        SELECT DISTINCT CAST(hadm_id AS BIGINT) AS hadm_id, 1 AS has_infection
        FROM heart_microbiologyevents
        WHERE hadm_id IS NOT NULL
    ),

    -- Prior admissions count
    prior_admissions AS (
        SELECT
            h.hadm_id,
            COUNT(prev.hadm_id) AS num_prior_admissions
        FROM hf_labeled h
        LEFT JOIN admissions prev
            ON h.subject_id = prev.subject_id
            AND prev.admittime < h.admittime
        GROUP BY h.hadm_id
    )

    SELECT
        h.hadm_id,
        h.subject_id,
        h.readmitted_30d,

        -- Demographics
        p.anchor_age AS age,
        p.gender,

        -- Admission info
        a.admission_type,
        a.insurance,
        a.marital_status,
        a.race,
        a.discharge_location,

        -- Length of stay
        DATE_DIFF('day', h.admittime, h.dischtime) AS length_of_stay,

        -- Comorbidities
        COALESCE(c.num_comorbidities, 0) AS num_comorbidities,

        -- Procedures
        COALESCE(pr.num_procedures, 0) AS num_procedures,

        -- Infection
        COALESCE(inf.has_infection, 0) AS has_infection,

        -- Prior admissions
        COALESCE(pa.num_prior_admissions, 0) AS num_prior_admissions

    FROM hf_labeled h
    JOIN patients p ON h.subject_id = p.subject_id
    JOIN admissions a ON h.hadm_id = a.hadm_id
    LEFT JOIN comorbidity_count c ON h.hadm_id = c.hadm_id
    LEFT JOIN procedure_count pr ON h.hadm_id = pr.hadm_id
    LEFT JOIN infection_flag inf ON h.hadm_id = inf.hadm_id
    LEFT JOIN prior_admissions pa ON h.hadm_id = pa.hadm_id
""")

# Verify
other_df = con.execute("SELECT * FROM other_features LIMIT 5").fetchdf()
print(f"Other features: {con.execute('SELECT COUNT(*) FROM other_features').fetchone()[0]} rows, {len(other_df.columns)} columns")
print(f"\nColumns: {list(other_df.columns)}")
display(other_df)

# Step 3F: Combine All Features into Final Model-Ready Table

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE model_features AS
    SELECT
        o.*,
        l.* EXCLUDE (hadm_id)
    FROM other_features o
    LEFT JOIN lab_features l ON o.hadm_id = CAST(l.hadm_id AS BIGINT)
""")

# Verify
mf = con.execute("SELECT * FROM model_features LIMIT 5").fetchdf()
print(f"Final dataset: {con.execute('SELECT COUNT(*) FROM model_features').fetchone()[0]} rows, {len(mf.columns)} columns")

print(f"\nLabel distribution:")
print(con.execute("""
    SELECT readmitted_30d, COUNT(*) AS count
    FROM model_features
    GROUP BY readmitted_30d
""").fetchdf().to_string(index=False))

print(f"\nMissing values (top 10):")
missing = con.execute("""
    SELECT * FROM (
        SELECT 'creatinine_first' AS col, COUNT(*) - COUNT(creatinine_first) AS missing FROM model_features
        UNION ALL SELECT 'sodium_first', COUNT(*) - COUNT(sodium_first) FROM model_features
        UNION ALL SELECT 'troponin_t_first', COUNT(*) - COUNT(troponin_t_first) FROM model_features
        UNION ALL SELECT 'inrpt_first', COUNT(*) - COUNT(inrpt_first) FROM model_features
        UNION ALL SELECT 'hemoglobin_first', COUNT(*) - COUNT(hemoglobin_first) FROM model_features
        UNION ALL SELECT 'marital_status', COUNT(*) - COUNT(marital_status) FROM model_features
        UNION ALL SELECT 'discharge_location', COUNT(*) - COUNT(discharge_location) FROM model_features
    ) ORDER BY missing DESC
""").fetchdf()
print(missing.to_string(index=False))

Joined the `other_features` with `lab_features` to create the unified `model_features` table. Verifies the final dataset dimensions (4,508 rows × 71 columns), confirms the class distribution (78.5% not readmitted vs. 21.5% readmitted), and checks for missing values. Key missing columns include Troponin T (~46% missing) and INR (~14% missing), which is expected since these tests are not ordered for every patient.

# Step 4: Data Preprocessing — Feature Engineering, Imputation, Encoding, and Train/Test Split


In [ ]:
import pandas as pd
import numpy as np

df = con.execute("SELECT * FROM model_features").fetchdf()

# 1. Add lab CHANGE features (last - first)
LABS = [
    'creatinine', 'urea_nitrogen', 'sodium', 'potassium', 'glucose',
    'hemoglobin', 'white_blood_cells', 'platelet_count', 'bicarbonate',
    'calcium_total', 'inrpt', 'ptt', 'troponin_t',
    'creatine_kinase_mb_isoenzyme'
]

for lab in LABS:
    df[f'{lab}_change'] = df[f'{lab}_last'] - df[f'{lab}_first']

print(f"After adding change features: {df.shape[0]} rows, {df.shape[1]} columns")

#  2. Fill missing values
# Numerical columns: fill with median
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols.remove('readmitted_30d')  # don't touch the label
num_cols.remove('hadm_id')
num_cols.remove('subject_id')

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Categorical columns: fill with 'UNKNOWN'
cat_cols = ['gender', 'admission_type', 'insurance', 'marital_status', 'race', 'discharge_location']
for col in cat_cols:
    df[col] = df[col].fillna('UNKNOWN')

print(f"\nMissing values after filling: {df.isnull().sum().sum()}")

# 3. Encode categorical columns
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f"After encoding: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns")

# 4. Train/Test Split
from sklearn.model_selection import train_test_split

# Separate features and label
drop_cols = ['hadm_id', 'subject_id', 'readmitted_30d']
X = df_encoded.drop(columns=drop_cols)
y = df_encoded['readmitted_30d']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n Final Split ")
print(f"Training set: {X_train.shape[0]} rows, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} rows, {X_test.shape[1]} features")
print(f"\nTrain label distribution:")
print(y_train.value_counts().to_string())
print(f"\nTest label distribution:")
print(y_test.value_counts().to_string())

Prepares the dataset for modeling through four preprocessing steps:
1. **Lab change features** — computes `last - first` values for all 14 labs to capture patient trajectory during hospitalization.
2. **Missing value imputation** — fills numerical columns with median values and categorical columns with 'UNKNOWN'.
3. **One-hot encoding** — converts 6 categorical features (gender, admission type, insurance, marital status, race, discharge location) into binary columns using `pd.get_dummies`.
4. **Stratified train/test split** — 80/20 split preserving the class ratio (~21.5% readmitted in both sets).

Final dimensions: Training set (3,606 rows × 137 features), Test set (902 rows × 137 features).

# Step 5 — Initial Model Training
# Step 5A: Baseline Model Training — Logistic Regression vs. XGBoost

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
import matplotlib.pyplot as plt

# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model 1: Logistic Regression

print("MODEL 1: LOGISTIC REGRESSION")

lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # handles class imbalance
    random_state=42
)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]

print(f"\nROC-AUC Score: {roc_auc_score(y_test, lr_proba):.4f}")
print(f"F1 Score: {f1_score(y_test, lr_pred):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=['Not Readmitted', 'Readmitted']))

# Model 2: XGBoost (Gradient Boosting)

print("MODEL 2: XGBOOST (Gradient Boosting)")

xgb = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)
xgb.fit(X_train, y_train)  # XGBoost doesn't need scaling
xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:, 1]

print(f"\nROC-AUC Score: {roc_auc_score(y_test, xgb_proba):.4f}")
print(f"F1 Score: {f1_score(y_test, xgb_pred):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, xgb_pred, target_names=['Not Readmitted', 'Readmitted']))

# Compare with ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_proba)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_proba)

axes[0].plot(lr_fpr, lr_tpr, label=f'Logistic Reg (AUC={roc_auc_score(y_test, lr_proba):.3f})')
axes[0].plot(xgb_fpr, xgb_tpr, label=f'XGBoost (AUC={roc_auc_score(y_test, xgb_proba):.3f})')
axes[0].plot([0,1], [0,1], 'k--', label='Random (AUC=0.500)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve Comparison')
axes[0].legend()

# Confusion Matrices
for i, (name, pred) in enumerate([('Logistic Regression', lr_pred), ('XGBoost', xgb_pred)]):
    cm = confusion_matrix(y_test, pred)
    axes[1].text(0.05, 0.7 - i*0.45,
                 f"{name}:\n  TN={cm[0][0]}  FP={cm[0][1]}\n  FN={cm[1][0]}  TP={cm[1][1]}",
                 fontsize=12, fontfamily='monospace',
                 transform=axes[1].transAxes)
axes[1].set_title('Confusion Matrices')
axes[1].axis('off')

plt.tight_layout()
plt.show()

Trains two baseline models on all 137 features:
- **Logistic Regression** with `class_weight='balanced'` to handle class imbalance — achieves ROC-AUC 0.6305 and F1 0.3911 for readmitted class.
- **Gradient Boosting (XGBoost)** with 200 estimators — achieves ROC-AUC 0.6065 but very low F1 0.1545 due to poor recall on the minority class.

Key finding: Logistic Regression consistently outperforms XGBoost on ROC-AUC, likely because `class_weight='balanced'` directly addresses the ~78/22 class imbalance. ROC curves and confusion matrices confirm LR captures more true readmissions.

# Step 5B: Improved Models with Feature Selection

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# 1. Feature Selection: Keep top 40 most relevant features
selector = SelectKBest(score_func=f_classif, k=40)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()].tolist()
print(f"Selected {len(selected_features)} features:")
print(selected_features)

#  2. Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)


# Model 1: Logistic Regression (improved)

print("\n" )
print("MODEL 1: LOGISTIC REGRESSION (Improved)")

lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    C=0.1,               # stronger regularization
    random_state=42
)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]

print(f"\nROC-AUC Score: {roc_auc_score(y_test, lr_proba):.4f}")
print(f"F1 Score: {f1_score(y_test, lr_pred):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=['Not Readmitted', 'Readmitted']))


# Model 2: XGBoost (improved with class weight)

print("MODEL 2: XGBOOST (Improved)")

# Calculate scale_pos_weight for imbalance
scale_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

xgb = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=3,           # shallower trees to prevent overfitting
    learning_rate=0.05,    # slower learning
    subsample=0.8,
    min_samples_leaf=20,   # prevent overfitting
    random_state=42
)

# Use sample weights to handle class imbalance
sample_weights = np.where(y_train == 1, scale_weight, 1.0)
xgb.fit(X_train_selected, y_train, sample_weight=sample_weights)
xgb_pred = xgb.predict(X_test_selected)
xgb_proba = xgb.predict_proba(X_test_selected)[:, 1]

print(f"\nROC-AUC Score: {roc_auc_score(y_test, xgb_proba):.4f}")
print(f"F1 Score: {f1_score(y_test, xgb_pred):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, xgb_pred, target_names=['Not Readmitted', 'Readmitted']))

# Compare with ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_proba)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_proba)

axes[0].plot(lr_fpr, lr_tpr, label=f'Logistic Reg (AUC={roc_auc_score(y_test, lr_proba):.3f})')
axes[0].plot(xgb_fpr, xgb_tpr, label=f'XGBoost (AUC={roc_auc_score(y_test, xgb_proba):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.500)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve Comparison')
axes[0].legend()

# Confusion Matrices
for i, (name, cm) in enumerate([
    ('Logistic Regression', confusion_matrix(y_test, lr_pred)),
    ('XGBoost', confusion_matrix(y_test, xgb_pred))
]):
    axes[1].text(0.05, 0.7 - i * 0.45,
                 f"{name}:\n  TN={cm[0][0]}  FP={cm[0][1]}\n  FN={cm[1][0]}  TP={cm[1][1]}",
                 fontsize=12, fontfamily='monospace',
                 transform=axes[1].transAxes)
axes[1].set_title('Confusion Matrices')
axes[1].axis('off')

plt.tight_layout()
plt.show()

Improves both models using `SelectKBest`  to select the top 40 most predictive features, reducing noise from irrelevant columns. Key results:
- **Logistic Regression (improved)** — ROC-AUC increases from 0.6305 → 0.6559, F1 improves to 0.4056. Feature selection was the single biggest improvement.
- **XGBoost (improved)** with sample weighting for class imbalance — ROC-AUC 0.6083, F1 0.3622.

Insight: Better features matter more than better algorithms. Logistic Regression remains the stronger model for this imbalanced, moderate-sized dataset.

# Step 5C: SMOTE Oversampling and Threshold Analysis

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, classification_report

# 1. SMOTE: Create synthetic readmitted samples
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", pd.Series(y_train_smote).value_counts().to_dict())

# 2. Train both models on SMOTE data

# Logistic Regression
lr_smote = LogisticRegression(max_iter=1000, C=0.1, random_state=42)
lr_smote.fit(X_train_smote, y_train_smote)
lr_proba_smote = lr_smote.predict_proba(X_test_scaled)[:, 1]

# XGBoost (use non-scaled SMOTE data)
X_train_smote_raw, y_train_smote_raw = smote.fit_resample(X_train_selected, y_train)
xgb_smote = GradientBoostingClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, min_samples_leaf=20, random_state=42
)
xgb_smote.fit(X_train_smote_raw, y_train_smote_raw)
xgb_proba_smote = xgb_smote.predict_proba(X_test_selected)[:, 1]

# 3. Results
print("\n" )
print("LOGISTIC REGRESSION + SMOTE")
lr_auc = roc_auc_score(y_test, lr_proba_smote)
print("ROC-AUC:", round(lr_auc, 4))
lr_pred_smote = (lr_proba_smote >= 0.5).astype(int)
print(classification_report(y_test, lr_pred_smote, target_names=['Not Readmitted', 'Readmitted']))

print("XGBOOST + SMOTE")
xgb_auc = roc_auc_score(y_test, xgb_proba_smote)
print("ROC-AUC:", round(xgb_auc, 4))
xgb_pred_smote = (xgb_proba_smote >= 0.5).astype(int)
print(classification_report(y_test, xgb_pred_smote, target_names=['Not Readmitted', 'Readmitted']))

# 4. Threshold Analysis

print("THRESHOLD ANALYSIS — Logistic Regression")
for thresh in [0.30, 0.35, 0.40, 0.45, 0.50]:
    preds = (lr_proba_smote >= thresh).astype(int)
    tp = ((preds == 1) & (y_test == 1)).sum()
    fp = ((preds == 1) & (y_test == 0)).sum()
    fn = ((preds == 0) & (y_test == 1)).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    print(f"Threshold={thresh:.2f}  Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}")

# 5. ROC Curve
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_proba_smote)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_proba_smote)

ax.plot(lr_fpr, lr_tpr, label='LR + SMOTE (AUC=' + str(round(lr_auc, 3)) + ')')
ax.plot(xgb_fpr, xgb_tpr, label='XGB + SMOTE (AUC=' + str(round(xgb_auc, 3)) + ')')
ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve - SMOTE Models')
ax.legend()
plt.tight_layout()
plt.show()

Applied SMOTE to balance the training set (777 → 2,829 readmitted samples). Results:
- **LR + SMOTE** — ROC-AUC 0.6448, F1 0.40 (slight drop vs. Step 5B, showing `class_weight='balanced'` was already effective).
- **XGBoost + SMOTE** — ROC-AUC 0.6091, F1 only 0.13 (SMOTE did not help XGBoost).

Threshold analysis reveals that lowering the classification threshold from 0.5 to 0.30 dramatically increases recall (97.9%) but reduces precision (23.1%), demonstrating the precision-recall tradeoff inherent in readmission prediction.

Conclusion: SMOTE provides minimal benefit when LR's built-in `class_weight='balanced'` already handles imbalance effectively.

# Step 5D: Hyperparameter Tuning with GridSearchCV and 5-Fold Cross-Validation

In [ ]:
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 5-Fold Cross Validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# Model 1: Logistic Regression — GridSearch
print("MODEL 1: LOGISTIC REGRESSION — Hyperparameter Tuning")

lr_params = {
    'C': [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': ['balanced']
}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    lr_params,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)
lr_grid.fit(X_train_scaled, y_train)

print("Best params:", lr_grid.best_params_)
print("Best CV ROC-AUC:", round(lr_grid.best_score_, 4))

# Test set performance
lr_best = lr_grid.best_estimator_
lr_proba_best = lr_best.predict_proba(X_test_scaled)[:, 1]
lr_pred_best = lr_best.predict(X_test_scaled)

print("Test ROC-AUC:", round(roc_auc_score(y_test, lr_proba_best), 4))
print("Test F1:", round(f1_score(y_test, lr_pred_best), 4))
print(classification_report(y_test, lr_pred_best, target_names=['Not Readmitted', 'Readmitted']))


# Model 2: Gradient Boosting — GridSearch

print("MODEL 2: GRADIENT BOOSTING — Hyperparameter Tuning")
print("(This may take 2-3 minutes...)")

# Calculate class weight
scale_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
sample_weights = np.where(y_train == 1, scale_weight, 1.0)

gb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 3, 4],
    'learning_rate': [0.01, 0.05, 0.1],
    'min_samples_leaf': [10, 20, 30],
    'subsample': [0.8]
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_params,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)
gb_grid.fit(X_train_selected, y_train, sample_weight=sample_weights)

print("Best params:", gb_grid.best_params_)
print("Best CV ROC-AUC:", round(gb_grid.best_score_, 4))

# Test set performance
gb_best = gb_grid.best_estimator_
gb_proba_best = gb_best.predict_proba(X_test_selected)[:, 1]
gb_pred_best = gb_best.predict(X_test_selected)

print("Test ROC-AUC:", round(roc_auc_score(y_test, gb_proba_best), 4))
print("Test F1:", round(f1_score(y_test, gb_pred_best), 4))
print(classification_report(y_test, gb_pred_best, target_names=['Not Readmitted', 'Readmitted']))


# Final Comparison

print("FINAL COMPARISON — ALL MODELS")
print(f"{'Model':<35} {'CV AUC':<12} {'Test AUC':<12} {'Test F1':<10}")
print(f"{'LR (Step 5B)':<35} {'N/A':<12} {'0.6129':<12} {'0.3605':<10}")
print(f"{'LR Tuned (GridSearch)':<35} {round(lr_grid.best_score_, 4):<12} {round(roc_auc_score(y_test, lr_proba_best), 4):<12} {round(f1_score(y_test, lr_pred_best), 4):<10}")
print(f"{'GB (Step 5B)':<35} {'N/A':<12} {'0.5618':<12} {'0.2909':<10}")
print(f"{'GB Tuned (GridSearch)':<35} {round(gb_grid.best_score_, 4):<12} {round(roc_auc_score(y_test, gb_proba_best), 4):<12} {round(f1_score(y_test, gb_pred_best), 4):<10}")

# ROC Curve
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
for name, proba in [
    ('LR Tuned', lr_proba_best),
    ('GB Tuned', gb_proba_best)
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=name + ' (AUC=' + str(round(auc, 3)) + ')')

ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Tuned Models')
ax.legend()
plt.tight_layout()
plt.show()

Performed exhaustive hyperparameter tuning using `GridSearchCV` with 5-fold stratified cross-validation:
- **Logistic Regression** — Best params: C=0.05, L1 penalty, balanced class weights. CV ROC-AUC: 0.6248, Test ROC-AUC: **0.6546**, Test F1: **0.40**. This is our best and final model.
- **Gradient Boosting** — Best params: learning_rate=0.01, max_depth=2, 200 estimators. CV ROC-AUC: 0.6092, Test ROC-AUC: 0.6434, Test F1: 0.385.

**Final Comparison Summary:**
| Step | Model | CV AUC | Test AUC | Test F1 |
|------|-------|--------|----------|---------|
| 5A | LR Baseline | N/A | 0.5894 | 0.3462 |
| 5B | LR + Feature Selection | N/A | 0.6559 | 0.4056 |
| 5C | LR + SMOTE | N/A | 0.6448 | 0.40 |
| 5D | **LR Tuned (GridSearch)** | **0.6248** | **0.6546** | **0.40** |

Our final AUC of 0.61–0.65 is consistent with published MIMIC-based HF readmission studies (typically 0.60–0.68), confirming our methodology is sound and the moderate performance reflects the inherent difficulty of readmission prediction from structured EHR data alone.